In [58]:
import numpy as np
import pandas as pd
import re

## <center> 1. Предобработка таблицы Игредиенты

In [110]:
data = pd.read_csv('data/ingredients.csv', sep=';')
display(data.head(3))

,id,name,packing,store,price,type
0,1,сахар,1 кг,лента,62.99,продукты
1,2,печенье на декор,1 упаковка,лента,89.99,декор
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты


>Задача: получить столбец с ценой за 100г/100мл ингридиента
Для этого: 
* для упаковки "_ кг" привести к цене за 100 г 
* для упаковки "_ л" привести к цене за 100 мл
* для упаковки "_ штук" привести к цене за 1 штуку
* для упаковки "1 упаковка" оставить как есть
* для упаковки "_ кг" привести к цене за 100 г

In [111]:
def to_gramm(string):
    '''
    На вход функия получает строку с описанием фасовки продукта
    Если десятичные знаки разделены "," - меняем на "."
    На выходе получаем вес с граммах/мл
    '''
    string = string.split(' ')

    if string[1] == 'кг':
        if ',' in string[0]:
            return float(string[0].replace(',','.')) * 1000
        else:
            return float(string[0]) * 1000
    elif string[1] == 'л':
            if ',' in string[0]:
                return float(string[0].replace(',','.')) * 1000
            else:
                return float(string[0]) * 1000
    else:
        if ',' in string[0]:
            return float(string[0].replace(',','.'))
        else:
            return float(string[0])

In [112]:
def units(string):
    '''
    На вход функция получает строку
    На выходе получаем сокращенное обозначение фасовки ("г","мл","шт","уп")
    '''
    string = string.split(' ')

    if string[1] == 'кг' or string[1] == 'г':
        return 'г'
    elif string[1] == 'л' or string[1] == 'мл':
        return 'мл'
    elif string[1] == 'штук' or string[1] == 'шт' or string[1] == 'лист':
        return 'шт'
    elif string[1] == 'упаковка':
        return 'уп'
    else:
        pass


In [113]:
data['units'] = data['packing'].apply(units)
data['package weight'] = data['packing'].apply(to_gramm)

In [105]:
# Создаем признак 'price for unit' - цена за единицу измерения. Заполняем его нулями
data['price for unit'] = 0
data

,id,name,packing,store,price,type,units,package weight,price for unit
0,1,сахар,1 кг,лента,62.99,продукты,г,1000.0,0
1,2,печенье на декор,1 упаковка,лента,89.99,декор,уп,1.0,0
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты,мл,370.0,0
3,4,малина и голубика в сердце,1 упаковка,перекресток,579.99,декор,уп,1.0,0
4,5,яйцо с1 окское,10 шт,лента,109.99,продукты,шт,10.0,0
...,...,...,...,...,...,...,...,...,...
136,137,яйцо с0 365 дней,10 штук,лента,99.99,продукты,шт,10.0,0
137,138,малина с/м,1 кг,ольга,1600.00,продукты,г,1000.0,0
138,139,какао,90 г,лента,50.00,продукты,г,90.0,0
139,140,сода,500 г,лента,45.00,продукы,г,500.0,0


In [116]:
# Создаем маски для фильтрации внутри столбца
mask1 = (data['units'] == 'шт') | (data['units'] == 'уп')
mask2 = (data['units'] == 'г') | (data['units'] == 'мл')
# Считаем "price for unit" с учетом масок
data.loc[mask1,'price for unit'] = data.loc[mask1, 'price'] / data.loc[mask1,'package weight']
data.loc[mask2,'price for unit'] = (100 / data.loc[mask2, 'package weight']) * data.loc[mask2,'price']
# Удаляем уже ненужный признак "packing"
data = data.drop('packing', axis = 1)

In [117]:
# Задача выполнена
data.head()
data.to_csv('data/data.csv', index=False, sep=',')

---

## <center> Торт 3 шоколада

In [118]:
data = pd.read_csv('data/data.csv', sep=',')
recept_3_choco = pd.read_csv('data/3 шоколада.csv', sep = ',')

In [119]:
recept_3_choco

,product_name,weight
0,яйцо,2
1,сахар,60
2,мука,53
3,разрыхлитель,3
4,какао,15
5,шоколад темный,275
6,шоколад молочный,275
7,шоколад белый,275
8,сливки 33%,825
9,желатин,30


In [120]:
# Задача: создать столбец с названиями ингредиентов, как в таблице с рецептом
# Создаём столбец с NaN
data['3_choco'] = np.nan
# Для каждого продукта из рецепта
for product in recept_3_choco['product_name']:
    # Экранируем спецсимволы regex
    pattern = re.compile(re.escape(product))
    # Идём по индексам и значениям колонки 'name'
    for idx, name_value in data['name'].items():
        # Если совпадение найдено (search, не findall)
        if pattern.search(str(name_value)):
            # Записываем продукт в соответствующую ячейку
            data.at[idx, '3_choco'] = product

In [121]:
# Группируем таблицу для вывода минимальных цен на продукты, используемые в данном рецепте
# Выводим в сводной таблице те, столюцы, которые нужны для рассчета себестоимости и для планирования закупки
data.groupby(by = '3_choco')[['price for unit','units','store']].min()

,price for unit,units,store
3_choco,,,
вода,2.699800,мл,лента
желатин,179.980000,г,лента
какао,55.555556,г,лента
мука,5.799500,г,лента
разрыхлитель,54.000000,г,лента
сахар,6.199000,г,лента
сливки 33%,50.000000,мл,ольга
шоколад белый,149.987500,г,лента
шоколад молочный,112.487500,г,лента


In [122]:
# Объединяем таблицу с рецептом торта с таблицей ингредиентов и цен
price_for_3choco = recept_3_choco.merge(
    data.groupby(by = '3_choco')[['units','price for unit','store']].min(),
    left_on='product_name',
    right_on = '3_choco',
    how='left'
)


In [123]:
price_for_3choco

,product_name,weight,units,price for unit,store
0,яйцо,2,шт,7.599000,лента
1,сахар,60,г,6.199000,лента
2,мука,53,г,5.799500,лента
3,разрыхлитель,3,г,54.000000,лента
4,какао,15,г,55.555556,лента
5,шоколад темный,275,г,124.987500,лента
6,шоколад молочный,275,г,112.487500,лента
7,шоколад белый,275,г,149.987500,лента
8,сливки 33%,825,мл,50.000000,ольга
9,желатин,30,г,179.980000,лента


In [124]:
# Создаем маски для рассчета себестоимости с учетом того, что какоие-то продукты считаются поштучно, а что-то по весу
mask3 = price_for_3choco['units'] == 'шт'
mask4 = (price_for_3choco['units'] == 'г') | (price_for_3choco['units'] == 'мл')
price_for_3choco.loc[mask3,'result'] = price_for_3choco.loc[mask3, 'price for unit'] * price_for_3choco.loc[mask3,'weight']
price_for_3choco.loc[mask4,'result'] = price_for_3choco.loc[mask4, 'price for unit'] * price_for_3choco.loc[mask4,'weight'] / 100

In [125]:
print(' Себестоимость торта 3 шоколада весом 2 кг - ' + str(round(price_for_3choco['result'].sum(),2)) + ' рублей')

 Себестоимость торта 3 шоколада весом 2 кг - 1568.55 рублей


---

## <center> Нежность

In [126]:
data = pd.read_csv('data/data.csv', sep=',')
recept_caralis = pd.read_csv('data/нежность.csv', sep = ';')
recept_caralis

,product_name,weight
0,яйцо,9
1,сахар,300
2,ванилин,1
3,разрыхлитель,10
4,мука,150
5,сыр,440
6,сметана,600
7,желатин,30
8,вода,150


In [127]:
# Создаём столбец с NaN
data['caralis'] = np.nan
# Для каждого продукта из рецепта
for product in recept_caralis['product_name']:
    # Экранируем спецсимволы regex
    pattern = re.compile(re.escape(product))
    # Идём по индексам и значениям колонки 'name'
    for idx, name_value in data['name'].items():
        # Если совпадение найдено (search, не findall)
        if pattern.search(str(name_value)):
            # Записываем продукт в соответствующую ячейку
            data.at[idx, 'caralis'] = product

In [128]:
data.head(100)

,id,name,store,price,type,units,package weight,price for unit,caralis
0,1,сахар,лента,62.99,продукты,г,1000.0,6.299000,сахар
1,2,печенье на декор,лента,89.99,декор,уп,1.0,89.990000,NaN
2,3,сгущенка рогачев,лента,164.99,продукты,мл,370.0,44.591892,NaN
3,4,малина и голубика в сердце,перекресток,579.99,декор,уп,1.0,579.990000,NaN
4,5,яйцо с1 окское,лента,109.99,продукты,шт,10.0,10.999000,яйцо
...,...,...,...,...,...,...,...,...,...
95,96,сливки 33%,ольга,500.00,продукты,мл,1000.0,50.000000,NaN
96,97,шантипак,ольга,290.00,продукты,мл,1000.0,29.000000,NaN
97,98,шоколад молочный вес,ольга,1600.00,продукты,г,1000.0,160.000000,NaN
98,99,шоколад белый вес,ольга,800.00,продукты,г,500.0,160.000000,NaN


In [129]:
data.groupby(by = 'caralis')[['price for unit','units','store']].min()

,price for unit,units,store
caralis,,,
ванилин,466.000000,г,лента
вода,2.699800,мл,лента
желатин,179.980000,г,лента
мука,5.799500,г,лента
разрыхлитель,54.000000,г,лента
сахар,6.199000,г,лента
сметана,26.663333,г,лента
сыр,59.000000,г,лента
яйцо,7.599000,шт,лента


In [130]:
price_for_caralis = recept_caralis.merge(
    data.groupby(by = 'caralis')[['price for unit','units','store']].min(),
    left_on='product_name',
    right_on = 'caralis',
    how='left'
)

In [131]:
price_for_caralis

,product_name,weight,price for unit,units,store
0,яйцо,9,7.599000,шт,лента
1,сахар,300,6.199000,г,лента
2,ванилин,1,466.000000,г,лента
3,разрыхлитель,10,54.000000,г,лента
4,мука,150,5.799500,г,лента
5,сыр,440,59.000000,г,лента
6,сметана,600,26.663333,г,лента
7,желатин,30,179.980000,г,лента
8,вода,150,2.699800,мл,лента


In [132]:
mask3 = price_for_caralis['units'] == 'шт'
mask4 = (price_for_caralis['units'] == 'г') | (price_for_caralis['units'] == 'мл')
price_for_caralis.loc[mask3,'result'] = price_for_caralis.loc[mask3, 'price for unit'] * price_for_caralis.loc[mask3,'weight']
price_for_caralis.loc[mask4,'result'] = price_for_caralis.loc[mask4, 'price for unit'] * price_for_caralis.loc[mask4,'weight'] / 100

In [133]:
print(' Себестоимость торта Нежность весом 2 кг - ' + str(round(price_for_caralis['result'].sum(),2)) + ' рублей')


 Себестоимость торта Нежность весом 2 кг - 583.37 рублей


---

## <center> Медовик

In [150]:
data = pd.read_csv('data/data.csv', sep=',')
recept_medovik = pd.read_csv('data/медовик.csv', sep = ';')
recept_medovik

,product_name,weight
0,сахар,337
1,масло,150
2,мед,100
3,яйцо,3
4,сода,8
5,уксус,30
6,мука,800
7,сливки 33%,500
8,шантипак,150
9,сыр,500


In [151]:
# Создаём столбец с NaN
data['medovik'] = np.nan
# Для каждого продукта из рецепта
for product in recept_medovik['product_name']:
    # Экранируем спецсимволы regex
    pattern = re.compile(re.escape(product))
    # Идём по индексам и значениям колонки 'name'
    for idx, name_value in data['name'].items():
        # Если совпадение найдено (search, не findall)
        if pattern.search(str(name_value)):
            # Записываем продукт в соответствующую ячейку
            data.at[idx, 'medovik'] = product

In [152]:
data

,id,name,store,price,type,units,package weight,price for unit,medovik
0,1,сахар,лента,62.99,продукты,г,1000.0,6.299000,сахар
1,2,печенье на декор,лента,89.99,декор,уп,1.0,89.990000,NaN
2,3,сгущенка рогачев,лента,164.99,продукты,мл,370.0,44.591892,NaN
3,4,малина и голубика в сердце,перекресток,579.99,декор,уп,1.0,579.990000,NaN
4,5,яйцо с1 окское,лента,109.99,продукты,шт,10.0,10.999000,яйцо
...,...,...,...,...,...,...,...,...,...
136,137,яйцо с0 365 дней,лента,99.99,продукты,шт,10.0,9.999000,яйцо
137,138,малина с/м,ольга,1600.00,продукты,г,1000.0,160.000000,NaN
138,139,какао,лента,50.00,продукты,г,90.0,55.555556,NaN
139,140,сода,лента,45.00,продукы,г,500.0,9.000000,сода


In [153]:
data.groupby(by = 'medovik')[['price for unit','units','store','package weight']].min()

,price for unit,units,store,package weight
medovik,,,,
масло,13.9990,г,лента,180.0
мед,31.9980,г,лента,500.0
мука,5.7995,г,лента,1000.0
сахар,6.1990,г,лента,1000.0
сливки 33%,50.0000,мл,ольга,1000.0
сода,9.0000,г,лента,500.0
сыр,59.0000,г,лента,220.0
уксус,24.0000,мл,лента,250.0
шантипак,29.0000,мл,ольга,1000.0


In [154]:
price_for_medovik = recept_medovik.merge(
    data.groupby(by = 'medovik')[['price for unit','units','store']].min(),
    left_on='product_name',
    right_on = 'medovik',
    how='left'
)

In [155]:
price_for_medovik

,product_name,weight,price for unit,units,store
0,сахар,337,6.1990,г,лента
1,масло,150,13.9990,г,лента
2,мед,100,31.9980,г,лента
3,яйцо,3,7.5990,шт,лента
4,сода,8,9.0000,г,лента
5,уксус,30,24.0000,мл,лента
6,мука,800,5.7995,г,лента
7,сливки 33%,500,50.0000,мл,ольга
8,шантипак,150,29.0000,мл,ольга
9,сыр,500,59.0000,г,лента


In [156]:
mask3 = price_for_medovik['units'] == 'шт'
mask4 = (price_for_medovik['units'] == 'г') | (price_for_medovik['units'] == 'мл')
price_for_medovik.loc[mask3,'result'] = price_for_medovik.loc[mask3, 'price for unit'] * price_for_medovik.loc[mask3,'weight']
price_for_medovik.loc[mask4,'result'] = price_for_medovik.loc[mask4, 'price for unit'] * price_for_medovik.loc[mask4,'weight'] / 100

In [157]:
print(' Себестоимость торта Медовик весом 2 кг - ' + str(round(price_for_medovik['result'].sum(),2)) + ' рублей')


 Себестоимость торта Медовик весом 2 кг - 739.5 рублей


---